# mask

> Masking utilities used in measuring the ICL in images of galaxy clusters.

In [ ]:
# | default_exp mask

In [ ]:
# | exporti

from warnings import catch_warnings, filterwarnings

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from astropy.convolution import Gaussian2DKernel, convolve
from astropy.io import fits
from astropy.visualization import AsinhStretch, ImageNormalize
from photutils.background import Background2D, MedianBackground
from photutils.segmentation import deblend_sources, detect_sources, detect_threshold
from photutils.utils import circular_footprint
from scipy.ndimage import binary_dilation, binary_erosion
from scipy.stats import median_abs_deviation

from nicl.filter import sampled_mad_filter
from nicl.utilities import get_img_centre_pixel

In [ ]:
# | export


def create_bcg_mask(
    image: np.ndarray,  # input image data to mask
    sb_threshold,  # surface brightness threshold for the BCG mask
    bcg_pos=None,  # position of the BCG, if None assume the image centre
    wcs=None,  # WCS for converting a `bcg_pos` supplied as a SkyCoord to pixels
):  # the BCG image mask
    """Create a BCG mask for the specified `sb_threshold`.

    If `bcg_pos` can be provided as (x, y) pixels or a SkyCoord, in which case the `wcs` must be supplied.
    """
    segm = detect_sources(data=image, threshold=sb_threshold, npixels=5)
    if bcg_pos is None:
        bcg_pos = get_img_centre_pixel(image).astype(int)
    elif wcs is not None:
        bcg_pos = tuple(int(x) for x in bcg_pos.to_pixel(wcs))
    bcg_label = segm.data[*bcg_pos[::-1]]
    bcg_mask = segm.data == bcg_label
    return bcg_mask

In [ ]:
# | export


def fast_mask(
    image: np.ndarray,  # input image data to mask
    mask_params=None,  # mask parameters
    estimate_background=False,  # estimate the background from the image median
    max_repeat=10,  # maximum number of masking iterations
    relative_rms_tolerance=0.01,  # relative tolerance in the rms required to stop iterating
    return_threshold=True,  # return the threshold
):
    params = {
        "nsigma": 2.0,
        "background": 0.0,
        "erosion_iterations": 2,
        "dilation_radius": 3.0,
        "dilation_iterations": 3,
    }
    if mask_params:
        params.update(mask_params)
    mask = None
    bkg = params["background"]
    previous_rms = np.inf
    for i in range(max_repeat + 1):
        if mask is not None:
            masked_image = np.where(mask, np.nan, image)
        else:
            masked_image = image
        if estimate_background:
            bkg = np.nanmedian(masked_image)
        rms = median_abs_deviation(
            masked_image, axis=None, scale="normal", nan_policy="omit"
        )
        threshold = rms * params["nsigma"] + bkg
        mask = image > threshold
        if params["erosion_iterations"] > 0:
            mask = binary_erosion(mask, iterations=params["erosion_iterations"])
        if params["dilation_radius"] > 0 and params["dilation_iterations"] > 0:
            structure = circular_footprint(params["dilation_radius"])
            mask = binary_dilation(
                mask, structure=structure, iterations=params["dilation_iterations"]
            )
        if abs(rms - previous_rms) / rms < relative_rms_tolerance:
            break
        previous_rms = rms
    if return_threshold:
        return mask, threshold
    else:
        return mask

In [ ]:
# | export


def smooth_image(image, sigma):
    """Smooth `image` using Gaussian with standard deviation `sigma`."""
    kernel = Gaussian2DKernel(sigma)
    kernel.normalize()
    smoothed = convolve(image.data, kernel)
    return smoothed


def dilated_object_mask(
    segm: np.ndarray,  # a photutils SegmentationImage
    growth: float = 0.5,  # the relative padding around each source
    verbose: bool = False,  # print information about the iterations
):
    """Create an object mask from a segmentation image.

    The mask is grown by repeatedly dilating the image, each time removing successively larger segments,
    such that large segments are dilated more.

    The dilation structure initially increases in size, so that resulting masks are relatively circular,
    while avoiding large structures that result in slow operations.

    Using `growth=0.5` increases masks by 50% of their original radius,
    while `growth=1.0` doubles their size.

    """
    segm = segm.copy()
    obj_mask = binary_dilation(segm.data > 0)
    if growth > 0:
        size_min = 2
        while True:
            area_min = np.pi * size_min**2
            if verbose:
                print(f"current minimum size is {size_min} pixels")
            small_labels = segm.labels[segm.areas < area_min]
            if verbose:
                print(
                    f"removing {len(small_labels)} segments smaller than {area_min} pixels"
                )
            segm.remove_labels(small_labels)
            if verbose:
                print(f"{len(segm.labels)} segments remaining")
            if len(segm.labels) == 0:
                break
            radius = min(9, max(1, int(np.sqrt(size_min * growth))))
            iterations = max(1, int(size_min * growth / radius))
            if verbose:
                print(
                    f"applying {iterations} iterations of dilation with a radius of {radius}"
                )
            big_mask = binary_dilation(
                segm.data > 0,
                structure=circular_footprint(radius),
                iterations=iterations,
            )
            if verbose:
                print(f"new mask contains {big_mask.sum()} pixels")
            obj_mask = obj_mask | big_mask
            if verbose:
                print(f"currently {obj_mask.sum()} pixels masked")
            size_min *= 2
    return obj_mask


def create_object_mask(
    image: np.ndarray,  # input image data to mask
    bcg_mask: np.ndarray
    | None = None,  # the BCG mask, if provided objects whose centres overlap with this mask are removed
    growth=0.5,  # the relative padding around each source
    detection_params=None,  # detection parameters
    verbose=False,  # print information about the dilation iterations
):
    """Create an object mask."""
    params = {
        "nsigma": 2.0,
        "background": 0.0,
        "smooth_sigma": 1.0,
        "npixels": 5,
        "nlevels": 32,
        "contrast": 0.01,
        "bkg_box_size": None,
        "bkg_filter_size": 1,
    }
    if detection_params:
        params.update(detection_params)
    print("Estimating background and detection threshold")
    if params["bkg_box_size"] is not None:
        bkg_estimator = MedianBackground()
        bkg = Background2D(
            image,
            params["bkg_box_size"],
            filter_size=params["bkg_filter_size"],
            bkg_estimator=bkg_estimator,
        )
        image = image - bkg.background
        threshold = bkg.background_rms * params["nsigma"] + params["background"]
    else:
        bkg = params["background"]
        threshold = detect_threshold(
            image,
            nsigma=params["nsigma"],
            background=bkg,
        )
    print(f"Average threshold is {threshold.mean():.2f}")
    print("Smoothing image")
    detection_image = smooth_image(image, sigma=params["smooth_sigma"])
    print("Detecting sources")
    segm = detect_sources(
        data=detection_image,
        threshold=threshold,
        npixels=params["npixels"],
    )
    print(f"Deblending {segm.nlabels} sources")
    with catch_warnings():
        filterwarnings("ignore", ".*[Dd]eblending mode.*changed.*")
        segm_deblend = deblend_sources(
            detection_image,
            segm,
            npixels=params["npixels"],
            nlevels=params["nlevels"],
            contrast=params["contrast"],
            progress_bar=False,
        )
    if bcg_mask is not None:
        segm_deblend.remove_masked_labels(bcg_mask)
    print("Dilating mask")
    obj_mask = dilated_object_mask(segm_deblend, growth, verbose=verbose)
    return obj_mask, bkg, threshold

In [ ]:
# | export


def create_faint_mask(
    image: np.ndarray,  # input image data to mask
    object_mask: np.ndarray
    | None = None,  # the object mask, objects which are entirely covered by this mask are removed
    bcg_mask: np.ndarray
    | None = None,  # the BCG mask, objects whose centres overlap with this mask are removed
    growth=0.25,  # the relative padding around each source
    bkg_sigma=20,  # the sigma of the Gaussian weighted median filter used to estimate the background for detection
    detection_params=None,  # detection parameters
    verbose=False,  # print information about the dilation iterations
):
    """Create a faint mask by filtering the image before detection."""
    params = {
        "nsigma": 2.0,
        "smooth_sigma": 1.0,
        "npixels": 5,
        "nlevels": 128,
        "contrast": 0.001,
    }
    if detection_params:
        params.update(detection_params)
    print("Estimating background")
    bkg, rms = sampled_mad_filter(
        image.data, size=bkg_sigma * 5, nsample=None, gaussian_sigma=bkg_sigma
    )
    threshold = detect_threshold(
        image.data, nsigma=params["nsigma"], background=bkg, error=rms
    )
    print(f"Average threshold is {threshold.mean():.2f}")
    print("Smoothing image")
    detection_image = smooth_image(image, sigma=params["smooth_sigma"])
    print("Detecting sources")
    segm = detect_sources(
        data=detection_image,
        threshold=threshold,
        npixels=params["npixels"],
    )
    print(f"Deblending {segm.nlabels} sources")
    with catch_warnings():
        filterwarnings("ignore", ".*[Dd]eblending mode.*changed.*")
        segm_deblend = deblend_sources(
            detection_image,
            segm,
            npixels=params["npixels"],
            nlevels=params["nlevels"],
            contrast=params["contrast"],
            progress_bar=False,
        )
    segm_deblend.remove_masked_labels(bcg_mask)  # remove bcg region
    segm_deblend.remove_masked_labels(object_mask, partial_overlap=False)
    faint_mask = segm_deblend.data > 0
    print("Dilating mask")
    faint_mask = dilated_object_mask(segm_deblend, growth, verbose=verbose)
    return faint_mask, bkg, threshold

In [ ]:
# | export


def plot_mask(
    img,  # image to plot
    mask,  # mask to plot
    detail=False,  # if True, add a row of detail images
    detail_size=1000,  # size, in pixels, of the detail region
):
    """Create a plot displaying the original image, masked image and mask."""
    nrows = 2 if detail else 1
    fig, ax = plt.subplots(nrows, 3, figsize=(12, 4 * nrows), squeeze=False)
    rms = median_abs_deviation(img, axis=None, scale="normal", nan_policy="omit")
    norm = ImageNormalize(vmin=-2.5 * rms, vmax=10 * rms, stretch=AsinhStretch(0.1))
    cmap = mpl.colormaps.get_cmap("viridis")
    cmap.set_bad("black")
    ax[0, 0].imshow(img, norm=norm, origin="lower", interpolation="none", cmap=cmap)
    ax[0, 1].imshow(
        np.where(mask, np.nan, img),
        norm=norm,
        origin="lower",
        interpolation="none",
        cmap=cmap,
    )
    ax[0, 2].imshow(mask, origin="lower", interpolation="none", cmap="gray_r")
    ax[0, 0].set_title("Original Image")
    ax[0, 1].set_title("Masked Image")
    ax[0, 2].set_title("Object Mask")
    if detail:
        ci, cj = get_img_centre_pixel(img).astype(int)
        ax[0, 0].plot(
            [ci - detail_size, ci, ci, ci - detail_size, ci - detail_size],
            [cj, cj, cj - detail_size, cj - detail_size, cj],
            "w-",
        )
        ax[0, 0].set_ylabel("Full Image")
        ax[1, 0].set_ylabel("Detail")
        img = img[ci - detail_size : ci, cj - detail_size : cj]
        mask = mask[ci - detail_size : ci, cj - detail_size : cj]
        ax[1, 0].imshow(img, norm=norm, origin="lower", interpolation="none", cmap=cmap)
        ax[1, 1].imshow(
            np.where(mask, np.nan, img),
            norm=norm,
            origin="lower",
            interpolation="none",
            cmap=cmap,
        )
        ax[1, 2].imshow(mask, origin="lower", interpolation="none", cmap="gray_r")
    for a in ax.flat:
        a.xaxis.set_ticks([])
        a.yaxis.set_ticks([])
    plt.tight_layout()


def plot_background(
    img,  # original image to plot
    bkg,  # background image to plot
    threshold,  # threshold image to plot
):
    """Create a plot displaying the original image, background and threshold."""
    fig, ax = plt.subplots(1, 3, figsize=(12, 4))
    rms = median_abs_deviation(img, axis=None, scale="normal", nan_policy="omit")
    norm = ImageNormalize(vmin=-2.5 * rms, vmax=10 * rms, stretch=AsinhStretch(0.1))
    ax[0].imshow(img, norm=norm, origin="lower")
    if isinstance(bkg, Background2D):
        ax[1].imshow(bkg.background, norm=norm, origin="lower")
        bkg.plot_meshes(ax=ax[0], outlines=True, marker="+", color="white", alpha=0.75)
    else:
        ax[1].imshow(bkg, norm=norm, origin="lower")
    ax[2].imshow(threshold, norm=norm, origin="lower")
    ax[0].set_title("Original Image")
    ax[1].set_title("Background")
    ax[2].set_title("Threshold")
    for a in ax:
        a.xaxis.set_ticks([])
        a.yaxis.set_ticks([])
    plt.tight_layout()

In [ ]:
def write_mask(
    image_filename,  # full filename of the image to which the mask corresponds
    mask_name,  # name of the mask
    mask_data,  # the mask data to write
):
    """Save `mask_data` as a FITS file, at the same `image_filename` but with `name` appended."""
    fn = image_filename.replace(".fits", f"_{mask_name}.fits")
    fits.writeto(fn, mask_data.astype("uint8"), overwrite=True)

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()